In [13]:
import sys
# Add the repository path to sys.path so Python can find the 'data' package
sys.path.append('/content/unstructuredata_tfidf_repo')

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from src.load_data import load_documents

# Step 1: Load data
titles, docs = load_documents()

# Step 2: Build the TF-IDF Matrix
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(docs)

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Vocabulary Size: {len(vectorizer.get_feature_names_out())}")

TF-IDF Matrix Shape: (10, 537)
Vocabulary Size: 537


In [14]:

# Step 3: Inspect the Matrix as a DataFrame
feature_names = vectorizer.get_feature_names_out()
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), index=titles, columns=feature_names)

print("\nSample of the TF-IDF Matrix (first 3 documents, first 10 terms):")
print(df_tfidf.iloc[:3, :10])

print("\nTop-5 TF-IDF terms for each document:")
for title in titles:
    row = df_tfidf.loc[title]
    top5 = row.nlargest(5)
    print(f"- {title}: {', '.join([f'{word} ({score:.3f})' for word, score in top5.items()])}")



Sample of the TF-IDF Matrix (first 3 documents, first 10 terms):
                      ability  accelerate  access  accountability  accuracy  \
AI in Healthcare     0.000000         0.0     0.0          0.1246    0.1246   
Climate Change       0.000000         0.0     0.0          0.0000    0.0000   
Basketball Strategy  0.130137         0.0     0.0          0.0000    0.0000   

                     accurate   actions  activity  adaptation  add  
AI in Healthcare     0.000000  0.000000       0.0    0.000000  0.0  
Climate Change       0.124795  0.124795       0.0    0.124795  0.0  
Basketball Strategy  0.000000  0.000000       0.0    0.000000  0.0  

Top-5 TF-IDF terms for each document:
- AI in Healthcare: ai (0.374), healthcare (0.249), use (0.165), accountability (0.125), accuracy (0.125)
- Climate Change: change (0.250), emissions (0.250), climate (0.212), energy (0.212), accurate (0.125)
- Basketball Strategy: ability (0.130), analytics (0.130), basketball (0.130), breaks (0.130)

In [15]:

# Step 4: Query Similarity Challenge
def search_query(query: str, vectorizer: TfidfVectorizer, tfidf_matrix, titles):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    results = sorted(zip(titles, similarities), key=lambda x: x[1], reverse=True)
    return results

queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet"
]

print("\nQuery Search Results:")
for q in queries:
    print(f"\nQuery: '{q}'")
    ranks = search_query(q, vectorizer, tfidf_matrix, titles)
    for rank, (title, score) in enumerate(ranks[:3], 1):
        print(f"  {rank}. {title} (Similarity: {score:.4f})")


Query Search Results:

Query: 'renewable energy and climate policy'
  1. Climate Change (Similarity: 0.3235)
  2. Economics Inflation (Similarity: 0.0952)
  3. Gardening Vegetables (Similarity: 0.0485)

Query: 'password security and phishing'
  1. Cybersecurity (Similarity: 0.1764)
  2. AI in Healthcare (Similarity: 0.0000)
  3. Climate Change (Similarity: 0.0000)

Query: 'healthy sleep exercise and diet'
  1. Public Health (Similarity: 0.2849)
  2. Gardening Vegetables (Similarity: 0.0447)
  3. AI in Healthcare (Similarity: 0.0000)


In [16]:
# Step 5: Optional Challenge
custom_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)
custom_tfidf_matrix = custom_vectorizer.fit_transform(docs)
print(f"Custom TF-IDF Matrix Shape: {custom_tfidf_matrix.shape}")
print(f"Custom Vocabulary Size: {len(custom_vectorizer.get_feature_names_out())}")

custom_feature_names = custom_vectorizer.get_feature_names_out()
print("\nSample of custom vocabulary (first 20 features including bigrams):")
print(list(custom_feature_names)[:20])

Custom TF-IDF Matrix Shape: (10, 69)
Custom Vocabulary Size: 69

Sample of custom vocabulary (first 20 features including bigrams):
['adjust', 'based', 'careful', 'clean', 'climate', 'communication', 'data', 'decisions', 'defenses', 'depends', 'design', 'different', 'disease', 'economic', 'economic growth', 'efficiency', 'energy', 'essential', 'flavor', 'global']


In [17]:
# Step 6: Verify Their Work
# We will write the implementation into the repository's files to make sure the unit tests pass!
# Let's inspect the files first or directly run pytest to see current test state.
!pytest -q tests/test_tfidf_utils.py

...                                                                      [100%]
3 passed in 1.76s
